# Scripts for Evaluation
*Adjust paths as necessary* :)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import json
import cv2
import torch
from PIL import Image
from torchvision import transforms as T

from mpl_toolkits.mplot3d import Axes3D
import tikzplotlib

## Training and Validation Loss

In [ ]:
# Get training loss data
df = pd.read_csv("DS/log/DAVE2-SNGP-GPU-10/train_loss_array-3.txt", sep=" ", header=None)
cols = ["epoch","accel_loss","theta_loss","brake_loss","loss"] 
df.columns = cols
print(df.head())

In [ ]:
# Get validation loss data
val = pd.read_csv("DS/log/DAVE2-SNGP-GPU-10/val_loss_array-3.txt", sep=" ", header=None)
cols_val = ["epoch", "loss", "test likelihood"] 
val.columns = cols_val
print(val.tail())

In [ ]:
# Plot losses
plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
plt.title("losses")
plt.plot(df["epoch"], df["loss"], label="loss")
#plt.plot(df["epoch"], df["accel loss"], label="acceleration loss")
#plt.plot(df["epoch"], df["steer loss"], label="steering loss")
plt.xlabel("epoch")
plt.ylabel("mse loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.title("means")
sns.lineplot(data=df, x=df["epoch"], y=df["loss"], label="loss")
sns.lineplot(data=df, x=df["epoch"], y=df["theta_loss"], label="theta_loss")
sns.lineplot(data=df, x=df["epoch"], y=df["accel_loss"], label="accel_loss")
sns.lineplot(data=df, x=df["epoch"], y=df["brake_loss"], label="brake_loss")
sns.scatterplot(data=val, x=val["epoch"], y=val["loss"], label="loss")
sns.lineplot(data=val, x=val["epoch"], y=val["test likelihood"], label="test likelihood")
#sns.lineplot(data=df, x=df["epoch"], y=df["accel loss"], label="acceleration loss")
#sns.lineplot(data=df, x=df["epoch"], y=df["steer loss"], label="steering loss")
plt.ylabel("mse loss")


## Evaluation on CARLA data

In [ ]:
# Get predicted values and uncertainties after test run in CARLA
results = pd.DataFrame(columns=[
    "steer", 
    "uncertainty_steer", 
    "throttle", 
    "uncertainty_throttle", 
    "brake", 
    "position x",
    "position y",
    "compass",
    "speed"], dtype=float)
path = 'DS/data/results_DAVE2_SNGP-7/routes_town05_short_11_23_13_40_29/meta'
for _, _, files in os.walk(path):
    for file in sorted(files):
        f = open(path + "/" + file)
        r = pd.DataFrame([json.load(f)])
        results = pd.concat([results, r], ignore_index=True)
#results = results.drop(["position"], axis=1)
print(results.head(6))


In [ ]:
# Go through evaluation folder
routes_type = "val"
towns = ["town01"]
month = "11"
day = "29"
result_pattern = "routes_{}_{}_{}_{}" # town_route_month_day 
result_path = ""
data_folder = "DS/data/results_DAVE2_SNGP-10"

meta  = pd.DataFrame()
columns = [
    'steer', 
    'uncertainty_steer', 
    'throttle', 
    'uncertainty_throttle', 
    'brake', 
    'position x', 
    'position y', 
    'compass', 
    'speed',
    'route']
for town in towns:
    sub_folders = os.listdir(data_folder)
    sub_folders = sorted(list(sub_folders))
    result_file = result_pattern.format(town, routes_type, month, day)
    valid = [] 
    for sub in sub_folders:
        if result_file in str(sub):
            valid.append(sub + "/meta")
    for v in valid:
        folder = os.path.join(data_folder, v)
        meta_files = os.listdir(folder)
        meta_files = sorted(list(meta_files))
        for m in meta_files:
            with open(os.path.join(folder, m), 'r') as f:
                records = json.load(f)
                records['route'] = v
                if meta.empty:
                    meta = pd.DataFrame(records, index=[0], columns=columns)
                else:
                    meta = meta.append(records, ignore_index=True)

#print(meta)

In [ ]:
# Results for one CARLA scenario
cols_single = [
    "s_loss",
    "a_loss",
    "b_loss",
    "loss",
    "true_s",
    "true_a",
    "true_b",
    "pred_s",
    "pred_a",
    "pred_b",
    "var_s",
    "var_a",
    "img"]
path = 'DS/log/single-eval-10/train_loss_array.txt'
results_single = pd.read_csv(path, sep=" ", header=None)
results_single.columns = cols_single
print(results_single["fmap"].head(10))


In [ ]:
results_single = results_single.loc[results_single['img']=="routes_town05_short_01_04_12_17_28"]
plt.figure(1)
plt.plot(results_single.index, results_single["s_loss"], c="b", label="steer loss")
plt.plot(results_single.index, results_single["a_loss"], c="r", label="accel loss")
plt.plot(results_single.index, results_single["b_loss"], c="g", label="brake loss")
plt.legend()
plt.figure(2)
plt.plot(results_single.index, results_single["true_s"], c="m", label="steer true")
plt.plot(results_single.index, results_single["pred_s"], c="k", label="steer pred")
plt.legend()
plt.figure(3)
plt.plot(results_single.index, results_single["true_a"], c="m", label="accel true")
plt.plot(results_single.index, results_single["pred_a"], c="k", label="accel pred")
plt.legend()
plt.figure(4)
plt.plot(results_single.index, results_single["true_b"], c="m", label="brake true")
plt.plot(results_single.index, results_single["pred_b"], c="k", label="brake pred")
plt.legend()
plt.figure(5)
plt.plot(results_single.index, results_single["var_s"], c="b", label="var steer")
plt.plot(results_single.index, results_single["var_a"], c="r", label="var acc")
plt.legend()

## Comparing Full and Vanilla Model

In [ ]:
# Full vs Vanilla model
cols =[
    "s_loss",
    "a_loss",
    "b_loss",
    "loss",
    "true_s",
    "true_a",
    "true_b",
    "pred_s",
    "pred_a",
    "pred_b",
    "var_s",
    "var_a",
    "img",
] 

full = pd.read_csv("DS/log/single-eval-vanilla-vs-full/train_loss_array-full.txt", sep=" ", header=None)
vanilla = pd.read_csv("DS/log/single-eval-vanilla-vs-full/train_loss_array-vanilla.txt", sep=" ", header=None)
c5 = pd.read_csv("DS/log/single-eval-vanilla-vs-full/train_loss_array-fullc5.txt", sep=" ", header=None)
nospec = pd.read_csv("DS/log/single-eval-vanilla-vs-full/train_loss_array-fullnospec.txt", sep=" ", header=None)
full.columns = cols
vanilla.columns = cols
c5.columns = cols
nospec.columns = cols

cmap = sns.color_palette("rocket", as_cmap=True)
colors = cmap(np.linspace(0, 1, 5))

plt.figure(figsize=(15,5))
ax3 = plt.subplot(2, 1, 2)
plt.plot(full.index[:30], full["true_a"][:30], "k--", label="True")
plt.plot(full.index[:30], full["pred_a"][:30], c=colors[2], label="DS-c5-095")
plt.plot(full.index[:30], vanilla["pred_a"][:30], c='c', label="Vanilla")
plt.plot(full.index[:30], c5["pred_a"][:30], c=colors[1], label="DS-c5")
plt.plot(full.index[:30], nospec["pred_a"][:30], c=colors[3], label="DS-nospec")
plt.ylabel("acceleration", fontsize=8)
plt.xlabel("time step", fontsize=8)
plt.tick_params('x', labelsize=8)
plt.tick_params('y', labelsize=8)
plt.ylim(0, 1)
#plt.legend()
plt.tight_layout()

ax1 = plt.subplot(2, 1, 1, sharex=ax3)
plt.plot(full.index[:30], full["true_s"][:30], "k--", label="True")
plt.plot(full.index[:30], full["pred_s"][:30], c=colors[2], label="DS-c5-095")
plt.plot(full.index[:30], vanilla["pred_s"][:30], c='c', label="Vanilla")
plt.plot(full.index[:30], c5["pred_s"][:30], c=colors[1], label="DS-c5")
plt.plot(full.index[:30], nospec["pred_s"][:30], c=colors[3], label="DS-nospec")
plt.ylabel("steering angle", fontsize=8)
plt.tick_params('x', labelbottom=False)
plt.tick_params('y', labelsize=8)
plt.ylim(-0.6, 0.6)
plt.legend(fontsize=8)
plt.tight_layout()

#plt.savefig("DS/log/single-eval-vanilla-vs-full/vanilla_vs_full.svg", format="svg")

print("DS-c5-095 \t", np.mean(full["s_loss"]), "\t", np.mean(full["a_loss"]))
print("DS-c5 \t\t", np.mean(c5["s_loss"]), "\t", np.mean(c5["a_loss"]))
print("DS-nospec \t", np.mean(nospec["s_loss"]), "\t", np.mean(nospec["a_loss"]))
print("Vanilla \t", np.mean(vanilla["s_loss"]), "\t", np.mean(vanilla["a_loss"]))



## Domain Awareness

In [ ]:
# Scatter Plots to compare Domain Awareness
#full = pd.read_csv("DS/calibration/DS-c5-095.csv", sep=",", header=0)
noSing = pd.read_csv("DS/calibration/DS-c5-095_noSing.csv", sep=",", header=0) # only Boston no Singapore

print(noSing.columns)

l = ["carla", "nuscenes"]
plt.figure("means of recalibrated stddevs")
#plt.suptitle("means of recalibrated stddevs")
#sns.scatterplot(data=noSing, x="throttle_mean_gp", y="steer_mean_gp", hue="label", palette="rocket", s=30, alpha=0.7, linewidth=0)
sns.scatterplot(data=noSing, x="throttle_mean_uncali", y="steer_mean_uncali", hue="label", palette="rocket", s=30, alpha=0.7, linewidth=0)
#centroids = np.hstack([np.vstack([noSing[noSing["label"] == name][dir].mean()]for name in l) for dir in ["throttle_mean_gp", "steer_mean_gp"]])
centroids = np.hstack([np.vstack([noSing[noSing["label"] == name][dir].mean()]for name in l) for dir in ["throttle_mean_uncali", "steer_mean_uncali"]])
centroids = pd.DataFrame(centroids, columns=["x", "y"])
centroids["label"] = l
sns.scatterplot(data=centroids, x="x", y="y", hue="label", palette="rocket", s=400, marker="*")
plt.xlabel("Throttle mean uncalibrated stddev")
#plt.xlabel("Throttle mean recalibrated stddev")
plt.ylabel("Steer mean uncalibrated stddev")
#plt.ylabel("Steer mean recalibrated stddev")
plt.legend([None, "CARLA", "nuScenes", None])
#tikzplotlib.save("/home/carlas/Experiments/E2EDriving/UQ-E2E-AD/Thesis/pictures/means_wNorm_cali.tex")
#print(centroids)
#plt.savefig("/home/carlas/Experiments/E2EDriving/UQ-E2E-AD/Thesis/pictures/means_wNorm.svg", format="svg")

#plt.figure("variance of recalibrated stddevs")
#plt.suptitle("variance of recalibrated stddevs")
#sns.scatterplot(data=noSing, x="throttle_var_gp", y="steer_var_gp", hue="label", palette="rocket", s=50, alpha=0.8)
#centroids = np.hstack([np.vstack([noSing[noSing["label"] == name][dir].mean()]for name in l) for dir in ["throttle_var_gp", "steer_var_gp"]])
#centroids = pd.DataFrame(centroids, columns=["x", "y"])
#centroids["label"] = l
#sns.scatterplot(data=centroids, x="x", y="y", hue="label", palette="rocket", s=400, marker="*")
#plt.savefig("/home/carlas/Experiments/E2EDriving/UQ-E2E-AD/Thesis/pictures/vars_wNorm_cali.png")
#tikzplotlib.save("/home/carlas/Experiments/E2EDriving/UQ-E2E-AD/Thesis/pictures/vars_wNorm_cali.tex")

# https://stackoverflow.com/questions/62642878/how-to-plot-the-clusters-centroids-using-seaborn

#print(centroids)
#plt.figure()
#sns.pairplot(noSing, hue="label", palette="rocket")
#print(noSing.tail(), noSing.head())

print("NLL steering carla: \t", np.mean(noSing[noSing["label"] == "carla"]["steer_nll_uncali"]))
print("NLL steering nuscenes: \t", np.mean(noSing[noSing["label"] == "nuscenes"]["steer_nll_uncali"]))
print("NLL throttle carla: \t", np.mean(noSing[noSing["label"] == "carla"]["throttle_nll_uncali"]))
print("NLL throttle nuscenes: \t", np.mean(noSing[noSing["label"] == "nuscenes"]["throttle_nll_uncali"]))




In [ ]:
# Decision boundary?
v = np.zeros_like(noSing["steer_mean_gp"])
border = 0.0000001*0.2
print(border)
for i in range(noSing.index.stop):
    if noSing["throttle_var_uncali"][i] < border:
        v[i] += 1
    if noSing["steer_var_uncali"][i] < border:
        v[i] += 1  
    if noSing["throttle_var_gp"][i] < border:
        v[i] += 1
    if noSing["steer_var_gp"][i] < border:
        v[i] += 1
    if noSing["label"][i] == "carla" and v[i] >= 2:
    #if noSing["label"][i] == "nuscenes" and v[i] == 0:
        print(i, v[i], noSing["name"][i])
        print(noSing["throttle_var_gp"][i], noSing["steer_var_gp"][i])

noSing["var"] = v

sns.histplot(noSing, x="var", bins=5, hue="label", palette="rocket")

## Datasets

In [ ]:
# Check if datasets are similar
c = pd.read_csv("DS/calibration/c_scenario_dist_noNorm.csv", sep=",", header=0)
n = pd.read_csv("DS/calibration/n_scenario_dist_noNorm.csv", sep=",", header=0)

c['label'] = ["carla"]*c.index.stop
n['label'] = ["nuscenes"]*n.index.stop

both = pd.concat([c, n], axis=0)

print(both.columns)
plot = 1
plt.figure(figsize=(20, 10))
for value in both.columns:
    if value != "label":
        plt.subplot(2, 3, plot)
        plt.title(f"c: ({round(c[value].mean(), 3)}, {round(c[value].std(), 3)}); n: ({round(n[value].mean(), 3)}, {round(n[value].std(), 3)})")
        sns.histplot(c, x=value, stat="percent", palette="colorblind", hue="label", kde=True)
        sns.histplot(n, x=value, stat="percent", palette="rocket", hue="label", kde=True)
        plot += 1

#print("carla: \n", c.mean(), c.std())
#print("nuscenes: \n", n.mean(), n.std())

#plt.figure(figsize=(20, 20))
#sns.pairplot(both, kind="kde", hue="label", palette="rocket")

### Image normalization and resizing

In [ ]:
# Image normalization
img_mean = torch.load("DS/calibration/carla_img_mean.pt")
img_std = torch.load("DS/calibration/carla_img_std.pt")

img_mean = img_mean.cpu().numpy()
img_std = img_std.cpu().numpy()

print(img_mean)
print(img_std)

#plt.figure("mean and std", figsize=(20,10))
#for i in range(3):
    #map = img_mean[i, :, :]
    #plt.subplot(3, 2, 2*i+1)
    #plt.title(f"mean ch{i+1},{np.min(map)}, {np.max(map)}")
    #plt.imshow(map)
    #map = img_std[i, :, :]
    #plt.subplot(3, 2, 2*i+2)
    #plt.title(f"std ch{i+1},{np.min(map)}, {np.max(map)}")
    #plt.imshow(map)

In [ ]:
# Resize nuScenes images to CARLA format
source = "/home/carlas/Experiments/NuScenesE2E/boston-seaport/day/scene-0062/rgb/"

for _, _, pic in os.walk(source):
    for i in pic:
        if i.endswith('.jpg'):
            image = Image.open(source + '/' + i)
            resized = image.resize((900, 256))
            resized = np.array(resized).reshape((256, 900, 3))
            
            plt.figure(i, figsize=(15, 10))
            plt.subplot(1, 2, 1)
            plt.imshow(resized)
            plt.subplot(1, 2, 2)
            #resized = resized / 255
            m = img_mean
            s = img_std
            norm = ((resized-m)/s)
            plt.imshow(norm)

## Calibration

In [ ]:
# Calibration plots
import netcal
from netcal.metrics import NLL, PinballLoss, QCE
from netcal.presentation import ReliabilityRegression
from netcal.regression import VarianceScaling, GPNormal

In [ ]:
plot_c = pd.read_csv("DS/calibration/plot_calibration_c.csv", sep=",", header=0)
plot_n = pd.read_csv("DS/calibration/plot_calibration_n.csv", sep=",", header=0)
print(plot_c.columns)

In [ ]:
nll = NLL()
sc = nll.measure((np.array(plot_c["plot_c_mean_steer"]), np.array(plot_c["plot_c_std_recali_steer"])), np.array(plot_c["plot_c_gt_steer"]), reduction='mean')
sn = nll.measure((np.array(plot_n["plot_n_mean_steer"]), np.array(plot_n["plot_n_std_recali_steer"])), np.array(plot_n["plot_n_gt_steer"]), reduction='mean')
tc = nll.measure((np.array(plot_c["plot_c_mean_throttle"]), np.array(plot_c["plot_c_std_recali_throttle"])), np.array(plot_c["plot_c_gt_throttle"]), reduction='mean')
tn = nll.measure((np.array(plot_n["plot_n_mean_throttle"]), np.array(plot_n["plot_n_std_recali_throttle"])), np.array(plot_n["plot_n_gt_throttle"]), reduction='mean')


print("NLL steering carla: \t", np.mean(sc), np.std(sc))
print("NLL steering nuscenes: \t", np.mean(sn), np.std(sn))
print("NLL throttle carla: \t", np.mean(tc), np.std(tc))
print("NLL throttle nuscenes: \t", np.mean(tn), np.std(tn))
print("---------------")

sc = nll.measure((np.array(plot_c["plot_c_mean_steer"]), np.array(plot_c["plot_c_std_uncali_steer"])), np.array(plot_c["plot_c_gt_steer"]), reduction='mean')
sn = nll.measure((np.array(plot_n["plot_n_mean_steer"]), np.array(plot_n["plot_n_std_uncali_steer"])), np.array(plot_n["plot_n_gt_steer"]), reduction='mean')
tc = nll.measure((np.array(plot_c["plot_c_mean_throttle"]), np.array(plot_c["plot_c_std_uncali_throttle"])), np.array(plot_c["plot_c_gt_throttle"]), reduction='mean')
tn = nll.measure((np.array(plot_n["plot_n_mean_throttle"]), np.array(plot_n["plot_n_std_uncali_throttle"])), np.array(plot_n["plot_n_gt_throttle"]), reduction='mean')


print("NLL unc steering carla: \t", np.mean(sc), np.std(sc))
print("NLL unc steering nuscenes: \t", np.mean(sn), np.std(sn))
print("NLL unc throttle carla: \t", np.mean(tc), np.std(tc))
print("NLL unc throttle nuscenes: \t", np.mean(tn), np.std(tn))
print("---------------")
print("Mean steering uncertainty carla: \t", plot_c["plot_c_std_recali_steer"].mean())
print("Mean steering uncertainty nuscenes: \t", plot_n["plot_n_std_recali_steer"].mean())
print("Mean throttle uncertainty carla: \t", plot_c["plot_c_std_recali_throttle"].mean())
print("Mean throttle uncertainty nuscenes: \t", plot_n["plot_n_std_recali_throttle"].mean())
print("---------------")
print("std steering uncertainty carla: \t", plot_c["plot_c_std_recali_steer"].std())
print("std steering uncertainty nuscenes: \t", plot_n["plot_n_std_recali_steer"].std())
print("std throttle uncertainty carla: \t", plot_c["plot_c_std_recali_throttle"].std())
print("std throttle uncertainty nuscenes: \t", plot_n["plot_n_std_recali_throttle"].std())
print("---------------")
print("Mean steering mse carla: \t", plot_c["plot_c_loss_steer"].mean())
print("Mean steering mse nuscenes: \t", plot_n["plot_n_loss_steer"].mean())
print("Mean throttle mse carla: \t", plot_c["plot_c_loss_throttle"].mean())
print("Mean throttle mse nuscenes: \t", plot_n["plot_n_loss_throttle"].mean())
print("---------------")
print("std steering mse carla: \t", plot_c["plot_c_loss_steer"].std())
print("std steering mse nuscenes: \t", plot_n["plot_n_loss_steer"].std())
print("std throttle mse carla: \t", plot_c["plot_c_loss_throttle"].std())
print("std throttle mse nuscenes: \t", plot_n["plot_n_loss_throttle"].std())

In [ ]:
#carla
quantiles = np.linspace(0.1, 0.9, 9)
diagram = ReliabilityRegression(quantiles=quantiles)
diagram.plot((plot_c["plot_c_mean_steer"], plot_c["plot_c_std_uncali_steer"]), plot_c["plot_c_gt_steer"], title_suffix="steer uncalibrated")#, tikz=True, filename="/home/carlas/Experiments/E2EDriving/UQ-E2E-AD/Thesis/pictures/plot_c_steer_uncali.tex")
diagram.plot((plot_c["plot_c_mean_steer"], plot_c["plot_c_std_recali_steer"]), plot_c["plot_c_gt_steer"], title_suffix="steer recalibrated")#, tikz=True, filename="/home/carlas/Experiments/E2EDriving/UQ-E2E-AD/Thesis/pictures/plot_c_steer_recali.tex")
diagram.plot((plot_c["plot_c_mean_throttle"], plot_c["plot_c_std_uncali_throttle"]), plot_c["plot_c_gt_throttle"], title_suffix="throttle uncalibrated")#, tikz=True, filename="/home/carlas/Experiments/E2EDriving/UQ-E2E-AD/Thesis/pictures/plot_c_throttle_uncali.tex")
diagram.plot((plot_c["plot_c_mean_throttle"], plot_c["plot_c_std_recali_throttle"]), plot_c["plot_c_gt_throttle"], title_suffix="throttle recalibrated")#, tikz=True, filename="/home/carlas/Experiments/E2EDriving/UQ-E2E-AD/Thesis/pictures/plot_c_throttle_recali.tex")


In [ ]:
#nuscenes
quantiles = np.linspace(0.1, 0.9, 9)
diagram = ReliabilityRegression(quantiles=quantiles)
diagram.plot((plot_n["plot_n_mean_steer"], plot_n["plot_n_std_uncali_steer"]), plot_n["plot_n_gt_steer"], title_suffix="steer uncalibrated")#, tikz=True, filename="/home/carlas/Experiments/E2EDriving/UQ-E2E-AD/Thesis/pictures/plot_n_steer_uncali.tex")
diagram.plot((plot_n["plot_n_mean_steer"], plot_n["plot_n_std_recali_steer"]), plot_n["plot_n_gt_steer"], title_suffix="steer recalibrated")#, tikz=True, filename="/home/carlas/Experiments/E2EDriving/UQ-E2E-AD/Thesis/pictures/plot_n_steer_recali.tex")
diagram.plot((plot_n["plot_n_mean_throttle"], plot_n["plot_n_std_uncali_throttle"]), plot_n["plot_n_gt_throttle"], title_suffix="throttle uncalibrated")#, tikz=True, filename="/home/carlas/Experiments/E2EDriving/UQ-E2E-AD/Thesis/pictures/plot_n_throttle_uncali.tex")
diagram.plot((plot_n["plot_n_mean_throttle"], plot_n["plot_n_std_recali_throttle"]), plot_n["plot_n_gt_throttle"], title_suffix="throttle recalibrated")#, tikz=True, filename="/home/carlas/Experiments/E2EDriving/UQ-E2E-AD/Thesis/pictures/plot_n_throttle_recali.tex")
